In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F


@dp.table(
    name="finguard.gold.detect_high_value_trans",
    comment="This table has fraud transaction details"
)
def detect_high_value_trans():
    transactions = spark.readStream.table("finguard.silver.transactions")
    customers = spark.read.table("finguard.silver.customers")

    joined_df = (
        transactions.join(customers, transactions.customer_id == customers.customer_id, "left")
        .filter(transactions.amount > customers.transaction_limit)
        .select(
            F.concat_ws("-", F.lit("ALERT"), transactions.transaction_id).alias("Alert_ID"),
            F.lit("High Value Transaction").alias("Alert_Type"),
            F.current_timestamp().alias("Alert_Timestamp"),
            transactions.transaction_id.alias("Transaction_ID"),
            transactions.amount.alias("Transaction_Amount"),
            transactions.transaction_timestamp.alias("Transaction_Date"),
            transactions.customer_id.alias("Customer_ID"),
            transactions.currency
        )
    )

    return joined_df